In [ ]:
!pip install transformers datasets seqeval evaluate accelerate conllu -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.5 MB/s eta 0:00:00


In [ ]:
import numpy as np
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import evaluate
from datasets import Dataset
import conllu

# --- FİLE NAMES
TRAIN_FILE = "train_id.conllu"
DEV_FILE = "dev_id.conllu"
TEST_FILE = "test_id.conllu"

MODEL_NAME = "indolem/indobert-base-uncased"

# 1. data preprocessing
def load_conllu_data(file_path):
    print(f"Reading: {file_path}...")
    with open(file_path, "r", encoding="utf-8") as f:
        data = conllu.parse(f.read())

    sentences = []
    labels = []
    unique_tags = set()

    for sentence in data:
        tokens = [token["form"] for token in sentence]
        tags = [token["upos"] for token in sentence]

        if None in tokens or None in tags:
            continue

        sentences.append(tokens)
        labels.append(tags)
        unique_tags.update(tags)

    return sentences, labels, sorted(list(unique_tags))

# upload files
print("processing the files...")
train_s, train_l, tags_train = load_conllu_data(TRAIN_FILE)
dev_s, dev_l, tags_dev = load_conllu_data(DEV_FILE)
test_s, test_l, tags_test = load_conllu_data(TEST_FILE)

# united the label list
label_list = sorted(list(set(tags_train) | set(tags_dev) | set(tags_test)))
print(f"labels ({len(label_list)} piece): {label_list}")

# Dataset create
def create_hf_dataset(sentences, labels):
    return Dataset.from_dict({"tokens": sentences, "ner_tags": labels})

train_dataset = create_hf_dataset(train_s, train_l)
val_dataset = create_hf_dataset(dev_s, dev_l)
test_dataset = create_hf_dataset(test_s, test_l)

# 2. Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

def align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                if label[word_idx] in label2id:
                    label_ids.append(label2id[label[word_idx]])
                else:
                    label_ids.append(-100)
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

print("Data is tokenized...")
tokenized_train = train_dataset.map(align_labels, batched=True)
tokenized_val = val_dataset.map(align_labels, batched=True)
tokenized_test = test_dataset.map(align_labels, batched=True)

# 3. Model setup
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label_list), id2label=id2label, label2id=label2id
)

# 4. Metrics
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# 5. train setting
training_args = TrainingArguments(
    output_dir="indobert_pos_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Başlat
print("Training begins")
trainer.train()

# Test Sonuçlarını Hemen Alalım
print("📊 Calculated the results...")
results = trainer.evaluate(tokenized_test)
print(results)

Dosyalar işleniyor...
Okunuyor: train_id.conllu...
Okunuyor: dev_id.conllu...
Okunuyor: test_id.conllu...
Etiketler (16 adet): ['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.281801,0.878981,0.881839,0.880408,0.912702
2,0.446300,0.262369,0.890012,0.891500,0.890756,0.920393
3,0.446300,0.260223,0.891325,0.891407,0.891366,0.920552


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NOUN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171:

📊 Test Sonuçları Hesaplanıyor...


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NOUN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NUM seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171:

{'eval_loss': 0.2576679289340973, 'eval_precision': 0.8968694273826036, 'eval_recall': 0.900885660264703, 'eval_f1': 0.898873057637889, 'eval_accuracy': 0.9216468590831919, 'eval_runtime': 2.6429, 'eval_samples_per_second': 210.751, 'eval_steps_per_second': 13.243, 'epoch': 3.0}
